In [ ]:
import pandas as pd
import numpy as np

# STEP 1: Load Raw Dataset
print("Loading datasets...")
raw_df = pd.read_csv('retail-orders-raw.csv')
print(f"Original shape: {raw_df.shape}")

# STEP 2: Identify & Handle Missing Values, Outliers & Data Types
print("\nCleaning data...")
df = raw_df.copy()

# Remove duplicates
df = df.drop_duplicates(subset=['order_id'])

# Standardize categorical text data
df['customer_segment'] = df['customer_segment'].str.title()
df['payment_status'] = df['payment_status'].str.title()

# Handle missing values
df['city'] = df['city'].fillna('Unknown')
df['discount_pct'] = df['discount_pct'].fillna(0.0)

# Correct data types and clean anomalies in 'quantity'
df['quantity'] = df['quantity'].replace('two', 2)
df['quantity'] = pd.to_numeric(df['quantity'])
df['quantity'] = df['quantity'].apply(lambda x: abs(x) if x < 0 else x)

# Cap maximum discount percentage at 100%
df['discount_pct'] = df['discount_pct'].apply(lambda x: 100.0 if x > 100.0 else x)

# Fix Order Date parsing and drop invalid dates
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')
df = df.dropna(subset=['order_date'])

# STEP 3: Feature Engineering
print("\nPerforming feature engineering...")

# Extract month and year
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month

# Calculate Financials
df['total_revenue'] = df['quantity'] * df['unit_price'] * (1 - (df['discount_pct'] / 100))
df['unit_cost'] = df['unit_price'] * 0.60
df['total_cost'] = df['quantity'] * df['unit_cost']
df['profit'] = df['total_revenue'] - df['total_cost']

# Profit Margin % (handling divisions by zero)
df['profit_margin_pct'] = np.where(df['total_revenue'] > 0, (df['profit'] / df['total_revenue']) * 100, 0)

# Round financial columns for clean output
df = df.round({'total_revenue': 2, 'unit_cost': 2, 'total_cost': 2, 'profit': 2, 'profit_margin_pct': 2})

# STEP 4: Export the Clean, Standardized Dataset
output_filename = 'clean_dataset.csv'
df.to_csv(output_filename, index=False)
print(f"\nSuccess! Clean dataset saved locally as: {output_filename}")
print(f"Final shape: {df.shape}")

# Display the before and after proof required by the assignment
display(df.head())